### 1.2 Формулировка

**Входы**

- Задание и результаты первого знакомства с данными

**Выходы**

- Что предсказываем, по каким данным и для кого
- Требования к данным и критерий оценивания

**Что делаем**

Собираем постановку задачи: что подаём модели, что хотим получить и как поймём, насколько хорошо она работает

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


##### Для кого строим прогноз

#### Как будем понимать задачу

Предположим, что заёмщик уже подал заявку на кредит Кредитор оценил её, назначил грейд `loan_grade` и предложил ставку `loan_int_rate`

Наша модель нужна инвестору, который решает, финансировать ли такой кредит Поэтому грейд и ставку можно использовать: в выбранный момент они уже известны Возможно, при своей оценке кредитор учёл больше сведений, чем есть у нас Это допущение о сценарии работы, а не установленный факт об источнике данных

Столбец `loan_status` показывает фактический исход: 0 — без дефолта, 1 — дефолт Будем предсказывать его по остальным столбцам Столбец, который предсказываем, называют **целевой переменной**

Это **бинарная классификация**: модель выбирает между двумя классами, то есть двумя вариантами исхода Сначала она выдаёт оценку риска, а затем по выбранной границе — **порогу** — переводим её в класс 0 или 1

#### Что подаём модели и что получаем

Разметка — это известные исходы прошлых кредитов в столбце `loan_status` По ним модель учится отличать дефолты от остальных случаев

| Что определяем | В нашем случае |
| --- | --- |
| Одна запись | Заявка на кредит |
| Вход | Все 11 признаков, кроме `loan_status` |
| Результат | Оценка риска, а после применения порога — класс 0 или 1 |
| Кто использует прогноз | Инвестор, который рассматривает финансирование кредита |
| Когда строим прогноз | Грейд и ставка уже назначены, фактический исход ещё неизвестен |

Алгоритм пока не выбираем: сначала разберём данные

#### Какие данные нужны

Для обучения нужен известный исход кредита — `loan_status` Сам исход не подаём модели как признак

Для новых заявок нужны те же 11 полей и те же единицы измерения Например, ставка записана в процентах, а доля кредита от дохода — числом от 0 до 1

Используем только то, что известно в момент прогноза В нашем сценарии грейд и ставка уже есть, а фактического исхода кредита ещё нет Прошлые дефолты должны относиться именно к прошлому, а не к будущим событиям

Пропуски, повторы и подозрительные значения проверим при EDA Период, за который автор определял дефолт, не указан: модель будет предсказывать метку в том виде, в котором она дана в источнике

#### Как будем оценивать модель

Метрика — численная оценка качества Основной выберем **Average Precision (AP)**: она оценивает, насколько хорошо модель поднимает кредиты с дефолтом к началу списка по риску AP меняется от 0 до 1; при сравнении на одних данных больше — лучше

Для выбора порога понадобятся ещё два понятия:

- **Precision, или точность** — какая доля отмеченных моделью дефолтов оказалась настоящими
- **Recall, или полнота** — какую долю всех настоящих дефолтов модель нашла

**F2** объединяет точность и полноту, придавая полноте больший вес Так мы отдаём приоритет обнаружению дефолтов Стоимости двух видов ошибок не заданы, поэтому финансовую выгоду этим критерием не оцениваем

| Что выбираем | По какому правилу |
| --- | --- |
| Модель | Наибольшая AP на validation |
| Порог | Наибольшая F2 на validation для выбранной модели |
| Итоговое качество | Проверка на test; дополнительные метрики и ошибки разберём в этапе 6 |

Для сравнения возьмём и **базовую модель Dummy**: она не использует признаки, а всем заявкам присваивает риск, равный доле дефолтов в train Минимальную требуемую метрику задание не задаёт